# Competition 3 Model
TEAM 05 | 112021130 黃偉寧 | 112062313 柯俊安 | 112060002 羅詠璿 | 112062308 蔡佳倩

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

## Config / Hyperparameters

In [ ]:
class config:
    # Image Configuration
    IMAGE_HEIGHT = 64
    IMAGE_WIDTH = 64
    IMAGE_CHANNEL = 3

    # Sample Configuration
    SAMPLE_ROW = 8
    SAMPLE_COL = 8
    SAMPLE_NUM = SAMPLE_ROW * SAMPLE_COL

    # Paths
    # You can change DATA_ROOT to point to your local dataset folder
    DATA_ROOT = './dataset' 
    DICTIONARY_DIR = os.path.join(DATA_ROOT, 'dictionary')
    DATASET_DIR = os.path.join(DATA_ROOT, 'dataset')
    CHECKPOINTS_DIR = './checkpoints/ckpts/'
    SAMPLES_DIR = './samples/'
    INFERENCE_DIR = './inference/'

    # Hyperparameters
    Config = {
        'HIDDEN_DIM': 64,
        'DENSE_DIM': 128,
        'Z_DIM': 100,
        'BATCH_SIZE': 128,
        'LEARNING_RATE_G': 1e-4,
        'LEARNING_RATE_D': 4e-4,
        'BETA_1': 0.0,
        'BETA_2': 0.9,
        'LAMBDA': 10,
        'NUM_EPOCH': 200,
        'NOISE_DECAY_LIMIT': 30,
        'TRAIN_FROM_LATEST_EPOCH': True,
        'PRINT_FREQ': 5,
        'SAVE_FREQ': 1
    }


## dataset.py

In [ ]:
from transformers import T5Tokenizer, T5EncoderModel
import torch

class TextEncoder:
    def __init__(self, dictionary_dir):
        self.word2Id_dict = dict(np.load(os.path.join(dictionary_dir, 'word2Id.npy')))
        self.id2word_dict = dict(np.load(os.path.join(dictionary_dir, 'id2Word.npy')))

    def indices_list_to_text_list(self, indices_list):
        text_list = []
        for indices in indices_list:
            word_list = []
            for idx in indices:
                if idx == self.word2Id_dict.get('<RARE>', -1):
                    continue
                if idx == self.word2Id_dict.get('<PAD>', -1):
                    break
                if idx in self.id2word_dict:
                    word_list.append(self.id2word_dict[idx])
            text = ' '.join(word_list)
            if len(text.strip()) != 0:
                text_list.append(text)
        return text_list

def get_t5_embeddings(texts, batch_size=32):
    """
    Generate T5 embeddings for a list of texts.
    Uses t5-base model.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading T5-base model on {device}...")
    
    tokenizer = T5Tokenizer.from_pretrained("t5-base")
    model = T5EncoderModel.from_pretrained("t5-base").to(device)
    model.eval()
    
    all_embeddings = []
    
    # Process in batches to avoid OOM
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        # outputs.last_hidden_state shape: (batch, seq_len, 768)
        # Mean Pooling to get sentence embedding
        # Mask padding tokens to avoid averaging them
        attention_mask = inputs['attention_mask'].unsqueeze(-1) # (batch, seq_len, 1)
        masked_output = outputs.last_hidden_state * attention_mask
        sum_embeddings = torch.sum(masked_output, dim=1)
        sum_mask = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        
        all_embeddings.append(mean_embeddings.cpu().numpy())
        
        if (i // batch_size) % 10 == 0:
            print(f"Processed {i}/{len(texts)} texts...")
            
    return np.concatenate(all_embeddings, axis=0)

def prepare_embeddings(dataset_dir, encoder, is_train=True):
    # Filename changed to reflect T5
    filename = 'train_data_embedding_t5.pkl' if is_train else 'test_data_embedding_t5.pkl'
    source_file = 'text2ImgData.pkl' if is_train else 'testData.pkl'
    
    save_path = os.path.join(dataset_dir, filename)
    
    if os.path.exists(save_path):
        print(f"Loading embeddings from {save_path}...")
        df = pd.read_pickle(save_path)
    else:
        print(f"Generating T5 embeddings for {source_file}...")
        df = pd.read_pickle(os.path.join(dataset_dir, source_file))
        
        if is_train:
            # For training, we might have multiple captions per image? 
            # The original code structure implies df['Captions'] is a list of lists of indices
            # We need to flatten this to encode, then restructure?
            # Or just encode row by row.
            
            # Let's stick to the original logic: Convert indices to text list
            df['Texts'] = df['Captions'].apply(lambda x: encoder.indices_list_to_text_list(x))
            
            # Since 'Texts' is a list of strings for each row, we need to handle this carefully.
            # T5 is heavier than SBERT, so we should batch properly.
            # Let's flatten all texts, encode them, and then put them back.
            
            all_texts = []
            row_lens = []
            for text_list in df['Texts']:
                all_texts.extend(text_list)
                row_lens.append(len(text_list))
                
            print(f"Total {len(all_texts)} captions to encode...")
            embeddings_flat = get_t5_embeddings(all_texts)
            
            # Reconstruct
            embeddings_col = []
            cursor = 0
            for length in row_lens:
                embeddings_col.append(embeddings_flat[cursor : cursor+length])
                cursor += length
                
            df['Embeddings'] = embeddings_col
            
        else:
            # Test data
            df['Texts'] = df['Captions'].apply(lambda x: encoder.indices_list_to_text_list([x]))
            
            # Flatten
            all_texts = []
            for text_list in df['Texts']:
                all_texts.extend(text_list)
                
            print(f"Total {len(all_texts)} test captions to encode...")
            embeddings_flat = get_t5_embeddings(all_texts)
            
            # Since test data usually has 1 caption per row (based on original code logic which took [0])
            # Original: df_test['Embeddings'] = df_test['Texts'].apply(lambda x: sbert.encode(x)[0])
            # We need to match that structure.
            
            df['Embeddings'] = list(embeddings_flat) # Each row gets one embedding (array)
            
        df.to_pickle(save_path)
    return df

def map_train(embedding, image_path):
    embedding = tf.cast(embedding, tf.float32)

    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=config.IMAGE_CHANNEL, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    
    # Resize and set shape
    img = tf.image.resize(img, size=[config.IMAGE_HEIGHT, config.IMAGE_WIDTH])
    img.set_shape([config.IMAGE_HEIGHT, config.IMAGE_WIDTH, config.IMAGE_CHANNEL])

    # Augmentation
    if tf.random.uniform([]) < 0.4:
        img = tf.image.flip_left_right(img)
    if tf.random.uniform([]) < 0.4:
        img = tf.image.random_brightness(img, max_delta=0.2)
        
    # Normalize to [-1, 1]
    img = img * 2 - 1

    return embedding, img

def map_test(embedding, index):
    embedding = tf.cast(embedding, tf.float32)
    return embedding, index

def get_train_dataset(df, batch_size):
    embedding_list = []
    image_path_list = []
    
    # Assuming image paths in dataframe might need adjustment if they are relative
    # But for now, we'll assume they are correct or absolute, or we fix them here
    # The original code had some specific logic for kaggle paths, we'll try to be generic
    
    for embeddings, image_path in zip(df['Embeddings'].values, df['ImagePath'].values):
        # Fix path if it starts with ./ and we are not in the right root
        if image_path.startswith('./'):
             # If path is relative like ./images/..., join with DATA_ROOT's parent or similar
             # For safety, let's assume DATA_ROOT points to where 'images' folder is or similar
             # Adjust this logic based on actual file structure
             full_path = os.path.join(config.DATA_ROOT, image_path[2:])
        else:
             full_path = image_path

        for embedding in embeddings:
            embedding_list.append(embedding)
            image_path_list.append(full_path)

    dataset = tf.data.Dataset.from_tensor_slices((embedding_list, image_path_list))
    dataset = dataset.map(map_train, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(len(embedding_list))
    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

def get_test_dataset(df, batch_size):
    embedding_list = df['Embeddings'].values.tolist()
    index_list = df['ID'].values.tolist()

    dataset = tf.data.Dataset.from_tensor_slices((embedding_list, index_list))
    dataset = dataset.map(map_test, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size, drop_remainder=False)
    return dataset


## model.py

In [ ]:
# model.py
conv_init = tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02)
batchnorm_init = tf.keras.initializers.RandomNormal(mean=1.0, stddev=0.02)

class FiLM(tf.keras.layers.Layer):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.gamma = tf.keras.layers.Dense(channels, kernel_initializer=conv_init)
        self.beta = tf.keras.layers.Dense(channels, kernel_initializer=conv_init)

    def call(self, x, cond):
        gamma = tf.reshape(self.gamma(cond), [-1, 1, 1, self.channels])
        beta = tf.reshape(self.beta(cond), [-1, 1, 1, self.channels])
        return x * (1 + gamma) + beta

def residual_scale(x):
    # Keep residual outputs numerically stable when merging branches
    return x * 0.70710678

class ResidualUpBlock(tf.keras.layers.Layer):
    def __init__(self, filters, apply_film=True, dropout_rate=0.0):
        super().__init__()
        self.apply_film = apply_film
        self.upsample = tf.keras.layers.UpSampling2D(size=2, interpolation='nearest')
        self.conv1 = tf.keras.layers.Conv2D(
            filters=filters,
            kernel_size=3,
            strides=1,
            padding='same',
            use_bias=False,
            kernel_initializer=conv_init
        )
        self.bn1 = tf.keras.layers.BatchNormalization(gamma_initializer=batchnorm_init)
        self.conv2 = tf.keras.layers.Conv2D(
            filters=filters,
            kernel_size=3,
            strides=1,
            padding='same',
            use_bias=False,
            kernel_initializer=conv_init
        )
        self.bn2 = tf.keras.layers.BatchNormalization(gamma_initializer=batchnorm_init)
        self.skip = tf.keras.layers.Conv2D(
            filters=filters,
            kernel_size=1,
            strides=1,
            padding='same',
            kernel_initializer=conv_init
        )
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.film = FiLM(filters) if apply_film else None

    def call(self, x, cond, training=False):
        shortcut = self.skip(self.upsample(x))

        h = self.upsample(x)
        h = self.conv1(h)
        h = self.bn1(h, training=training)
        h = tf.nn.relu(h)

        if self.film is not None:
            h = self.film(h, cond)

        h = self.dropout(h, training=training)
        h = self.conv2(h)
        h = self.bn2(h, training=training)
        h = tf.nn.relu(h)

        return residual_scale(h + shortcut)

class Generator(tf.keras.Model):
    def __init__(self):
        super(Generator, self).__init__()
        self.hparams = config.Config
        self.dim = self.hparams['HIDDEN_DIM']

        # Stronger text conditioning that later feeds FiLM in each block
        self.embedding_layers = tf.keras.Sequential([
            tf.keras.layers.Dense(units=self.hparams['DENSE_DIM']),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.LeakyReLU(0.2),
            tf.keras.layers.Dense(units=self.hparams['DENSE_DIM']),
            tf.keras.layers.LeakyReLU(0.2)
        ])

        self.latent_dense = tf.keras.Sequential([
            tf.keras.layers.Dense(units=self.dim * 8 * 4 * 4, use_bias=False, kernel_initializer=conv_init),
            tf.keras.layers.BatchNormalization(gamma_initializer=batchnorm_init),
            tf.keras.layers.ReLU()
        ])

        self.blocks = [
            ResidualUpBlock(self.dim * 4, apply_film=True, dropout_rate=0.05),
            ResidualUpBlock(self.dim * 2, apply_film=True, dropout_rate=0.05),
            ResidualUpBlock(self.dim * 1, apply_film=True, dropout_rate=0.05),
            ResidualUpBlock(self.dim // 2, apply_film=False, dropout_rate=0.0)
        ]

        self.to_rgb = tf.keras.layers.Conv2D(
            filters=config.Config.IMAGE_CHANNEL,
            kernel_size=3,
            strides=1,
            padding='same',
            activation=tf.keras.activations.tanh,
            kernel_initializer=conv_init
        )

    def call(self, embedding, noise_z, training=False):
        cond = self.embedding_layers(embedding)

        latent = tf.concat([noise_z, cond], axis=1)
        x = self.latent_dense(latent, training=training)
        x = tf.reshape(x, [-1, 4, 4, self.dim * 8])

        for block in self.blocks:
            x = block(x, cond, training=training)

        x = self.to_rgb(x)

        return x

class ResidualDownBlock(tf.keras.layers.Layer):
    def __init__(self, filters, dropout_rate=0.0):
        super().__init__()
        self.conv1 = tf.keras.layers.Conv2D(
            filters=filters,
            kernel_size=3,
            strides=2,
            padding='same',
            kernel_initializer=conv_init
        )
        self.conv2 = tf.keras.layers.Conv2D(
            filters=filters,
            kernel_size=3,
            strides=1,
            padding='same',
            kernel_initializer=conv_init
        )
        self.skip = tf.keras.layers.Conv2D(
            filters=filters,
            kernel_size=1,
            strides=2,
            padding='same',
            kernel_initializer=conv_init
        )
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        shortcut = self.skip(x)

        h = self.conv1(x)
        h = tf.nn.leaky_relu(h, 0.2)
        h = self.dropout(h, training=training)
        h = self.conv2(h)
        h = tf.nn.leaky_relu(h, 0.2)

        return residual_scale(h + shortcut)

class Discriminator(tf.keras.Model):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.hparams = config.Config
        self.dim = self.hparams['HIDDEN_DIM']

        # Project text embedding to match deepest feature size for projection trick
        self.embedding_layers = tf.keras.Sequential([
            tf.keras.layers.Dense(units=self.hparams['DENSE_DIM']),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.LeakyReLU(0.2),
            tf.keras.layers.Dense(units=self.dim * 8),
            tf.keras.layers.LeakyReLU(0.2)
        ])

        self.from_rgb = tf.keras.layers.Conv2D(
            filters=self.dim,
            kernel_size=3,
            strides=1,
            padding='same',
            kernel_initializer=conv_init
        )

        self.blocks = [
            ResidualDownBlock(self.dim * 2, dropout_rate=0.05),
            ResidualDownBlock(self.dim * 4, dropout_rate=0.05),
            ResidualDownBlock(self.dim * 8, dropout_rate=0.05),
            ResidualDownBlock(self.dim * 8, dropout_rate=0.0)
        ]

        self.final_conv = tf.keras.layers.Conv2D(
            filters=self.dim * 8,
            kernel_size=3,
            strides=1,
            padding='same',
            kernel_initializer=conv_init
        )
        self.global_pool = tf.keras.layers.GlobalAveragePooling2D()
        self.logit_dense = tf.keras.layers.Dense(1)

    def call(self, embedding, image, training=False):
        cond = self.embedding_layers(embedding)

        x = self.from_rgb(image)
        x = tf.nn.leaky_relu(x, 0.2)

        for block in self.blocks:
            x = block(x, training=training)

        x = self.final_conv(x)
        x = tf.nn.leaky_relu(x, 0.2)

        pooled = self.global_pool(x)
        logits = self.logit_dense(pooled)

        # Text-image interaction through projection discriminator term
        # Projection Discriminator (Miyato et al.)
        # Inner product of [Global Features] and [Condition Embedding]
        cond_score = tf.reduce_sum(pooled * cond, axis=1, keepdims=True)
        # Normalize by feature dimension (optional but good for stability)
        cond_score = cond_score / tf.cast(tf.shape(pooled)[1], tf.float32)

        output = logits + cond_score

        return tf.squeeze(output, axis=1)


## trainer.py

In [ ]:
class Trainer:
    def __init__(self):
        self.hparams = config.Config
        self.generator = Generator()
        self.discriminator = Discriminator()
        
        self.g_optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.hparams['LEARNING_RATE_G'],
            beta_1=self.hparams['BETA_1'],
            beta_2=self.hparams['BETA_2']
        )
        self.d_optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.hparams['LEARNING_RATE_D'],
            beta_1=self.hparams['BETA_1'],
            beta_2=self.hparams['BETA_2']
        )
        
        self.checkpoint = tf.train.Checkpoint(
            generator_optimizer=self.g_optimizer,
            discriminator_optimizer=self.d_optimizer,
            generator=self.generator,
            discriminator=self.discriminator
        )
        self.checkpoint_manager = tf.train.CheckpointManager(
            self.checkpoint, 
            config.CHECKPOINTS_DIR, 
            max_to_keep=20
        )
        
        # TensorBoard
        current_time = time.strftime("%Y%m%d-%H%M%S")
        log_dir = 'logs/gradient_tape/' + current_time
        self.summary_writer = tf.summary.create_file_writer(log_dir)

    def restore_checkpoint(self):
        if self.hparams['TRAIN_FROM_LATEST_EPOCH'] and self.checkpoint_manager.latest_checkpoint:
            self.checkpoint.restore(self.checkpoint_manager.latest_checkpoint)
            print(f'Restored from {self.checkpoint_manager.latest_checkpoint}')
            return int(self.checkpoint_manager.latest_checkpoint.split('-')[-1])
        return 0

    @tf.function
    def train_step_d(self, embedding, real_images, noise_decay):
        batch_size = tf.shape(real_images)[0]
        noise_z = tf.random.normal([batch_size, self.hparams['Z_DIM']])
        
        with tf.GradientTape() as tape:
            fake_images = self.generator(embedding, noise_z, training=True)
            
            # Instance Noise
            fake_images_noisy = fake_images + noise_decay * tf.random.normal(tf.shape(fake_images))
            real_images_noisy = real_images + noise_decay * tf.random.normal(tf.shape(real_images))
            
            real_logits = self.discriminator(embedding, real_images_noisy, training=True)
            fake_logits = self.discriminator(embedding, fake_images_noisy, training=True)
            
            d_loss = tf.reduce_mean(fake_logits) - tf.reduce_mean(real_logits)
            
            # Gradient Penalty
            alpha = tf.random.uniform([batch_size, 1, 1, 1], 0.0, 1.0)
            interpolated = alpha * real_images + (1 - alpha) * fake_images
            
            with tf.GradientTape() as gp_tape:
                gp_tape.watch(interpolated)
                gp_logits = self.discriminator(embedding, interpolated, training=True)
            
            grads = gp_tape.gradient(gp_logits, interpolated)
            grad_norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]))
            gp = tf.reduce_mean((grad_norm - 1.0) ** 2)
            
            total_d_loss = d_loss + self.hparams['LAMBDA'] * gp
            
        gradients = tape.gradient(total_d_loss, self.discriminator.trainable_variables)
        self.d_optimizer.apply_gradients(zip(gradients, self.discriminator.trainable_variables))
        
        return total_d_loss, d_loss, gp

    @tf.function
    def train_step_g(self, embedding, noise_decay):
        batch_size = tf.shape(embedding)[0]
        noise_z = tf.random.normal([batch_size, self.hparams['Z_DIM']])
        
        with tf.GradientTape() as tape:
            fake_images = self.generator(embedding, noise_z, training=True)
            fake_images_noisy = fake_images + noise_decay * tf.random.normal(tf.shape(fake_images))
            
            fake_logits = self.discriminator(embedding, fake_images_noisy, training=True)
            g_loss = -tf.reduce_mean(fake_logits)
            
        gradients = tape.gradient(g_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(gradients, self.generator.trainable_variables))
        
        return g_loss

    def train(self, dataset, epochs):
        start_epoch = self.restore_checkpoint()
        
        for epoch in range(start_epoch, epochs):
            start_time = time.time()
            print(f"Epoch {epoch+1}/{epochs}")
            
            # Noise decay
            if epoch < self.hparams['NOISE_DECAY_LIMIT']:
                noise_decay = 1.0 / (epoch + 1)
            else:
                noise_decay = 0.0
                
            # Training loop
            d_losses = []
            g_losses = []
            
            # WGAN-GP usually trains D more times than G
            n_critic = 7
            
            step = 0
            for batch in dataset:
                embedding, real_images = batch
                
                # Train Discriminator
                d_loss, wasserstein_d, gp = self.train_step_d(embedding, real_images, noise_decay)
                d_losses.append(d_loss)
                
                # Train Generator every n_critic steps
                if step % n_critic == 0:
                    g_loss = self.train_step_g(embedding, noise_decay)
                    g_losses.append(g_loss)
                
                step += 1
            
            # Logging
            avg_d_loss = np.mean(d_losses)
            avg_g_loss = np.mean(g_losses) if g_losses else 0
            
            with self.summary_writer.as_default():
                tf.summary.scalar('d_loss', avg_d_loss, step=epoch)
                tf.summary.scalar('g_loss', avg_g_loss, step=epoch)
            
            print(f"Time: {time.time()-start_time:.2f}s, D Loss: {avg_d_loss:.4f}, G Loss: {avg_g_loss:.4f}")
            
            if (epoch + 1) % self.hparams['SAVE_FREQ'] == 0:
                self.checkpoint_manager.save(checkpoint_number=epoch+1)


## Entry

In [ ]:
def main(mode, steps=None):
    # Setup environment
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
    
    # GPU Setup
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"Found {len(gpus)} GPUs")
        except RuntimeError as e:
            print(e)

    # Initialize Text Encoder
    print("Initializing Text Encoder...")
    encoder = dataset.TextEncoder(config.DICTIONARY_DIR)

    if mode == 'train':
        print("Preparing Training Data...")
        df_train = dataset.prepare_embeddings(config.DATASET_DIR, encoder, is_train=True)
        train_ds = dataset.get_train_dataset(df_train, config.Config['BATCH_SIZE'])
        
        print("Initializing Trainer...")
        trainer = Trainer()
        
        epochs = steps if steps else config.Config['NUM_EPOCH']
        print(f"Starting Training for {epochs} epochs...")
        trainer.train(train_ds, epochs)

    elif mode == 'inference':
        print("Preparing Test Data...")
        df_test = dataset.prepare_embeddings(config.DATASET_DIR, encoder, is_train=False)
        test_ds = dataset.get_test_dataset(df_test, config.Config['BATCH_SIZE'])
        
        print("Initializing Trainer (for Generator)...")
        trainer = Trainer()
        trainer.restore_checkpoint()
        
        if not os.path.exists(config.INFERENCE_DIR):
            os.makedirs(config.INFERENCE_DIR)
            
        print("Starting Inference...")
        count = 0
        for batch in test_ds:
            embeddings, indices = batch
            batch_size = tf.shape(embeddings)[0]
            
            # Generate noise
            noise_z = tf.random.normal([batch_size, config.Config['Z_DIM']])
            
            # Generate images
            fake_images = trainer.generator(embeddings, noise_z, training=False)
            
            # Save images
            for i in range(batch_size):
                # Rescale to [0, 1]
                img = (fake_images[i] + 1) / 2.0
                img = tf.clip_by_value(img, 0.0, 1.0).numpy()
                
                idx = indices[i]
                save_path = os.path.join(config.INFERENCE_DIR, f'inference_{idx:04d}.jpg')
                plt.imsave(save_path, img)
                count += 1
                
        print(f"Generated {count} images in {config.INFERENCE_DIR}")

In [ ]:
if __name__ == '__main__':
    # Configuration
    # MODE options: 'train', 'inference'
    MODE = 'train' 
    
    # STEPS: Number of epochs to train. Set to None to use config.NUM_EPOCH
    STEPS = 100
    
    main(MODE, STEPS)
